<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


# ROS 2 Names and Remapping

Running several instances of the same node requires a way to identify each instance and connect it to the intended topics. ROS 2 remapping adapts names at launch so one implementation can be reused in different graphs. We will name a second publisher, then move a publisher-subscriber pair to another topic and inspect both results.

## Prepare each terminal

Build once after source changes so all terminals use the current installed package:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
cd "$COLCON_WS"
colcon build --packages-select ros2_pubsub
```

Source the distribution and overlay in each terminal that runs a node or inspects the graph:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
cd "$COLCON_WS"
source install/setup.bash
```

Both activities use two terminals for running nodes and a third for observing them. Keeping inspection separate lets you query the graph while both processes remain active.

The initial node names are `talker` and `listener`, and their relative topic name is `chatter`. In the root namespace, that topic resolves to `/chatter`.

## Give a second publisher a distinct node name

Distinct names let node-specific tools identify each instance. ROS 2 permits duplicate names, but a query using the shared name can be ambiguous. Remap the second instance's name at launch.

Start the default publisher in one prepared terminal:

```bash
ros2 run ros2_pubsub my_publisher
```

In the second, start another instance with a distinct name:

```bash
ros2 run ros2_pubsub my_publisher --ros-args -r __node:=talker_two
```

Inspect from the third terminal:

```bash
ros2 node list
```

The list should show `/talker` and `/talker_two`, giving each discovered instance its own name for inspection. Both run the same publisher implementation. Stop them with `Ctrl-C` before the next activity.

## Remap both ends of a topic

Move the conversation from `/chatter` to `/greeting`. With earlier nodes stopped, run these commands in separate node terminals:

```bash
ros2 run ros2_pubsub my_publisher --ros-args -r chatter:=greeting
ros2 run ros2_pubsub my_subscriber --ros-args -r chatter:=greeting
```

Both ends need the same resolved topic name because it is part of their communication contract. Observe a message and inspect the connections from the third terminal:

```bash
ros2 topic echo --once /greeting
ros2 topic info --verbose /greeting
```

The echo shows that a message arrived on the remapped topic. The detailed information locates the publisher and subscriber endpoints on `/greeting`; together with the listener's output, these results show how launch-time names connect the unchanged implementations.

To observe continuously, use `ros2 topic echo /greeting` and stop it with `Ctrl-C`. Stop both nodes when you finish.

## Further reading

The ROS 2 Jazzy guide to [command-line node arguments](https://docs.ros.org/en/jazzy/How-To-Guides/Node-arguments.html) explains node-name and topic remapping. The ROS 2 design article on [topic and service names](https://design.ros2.org/articles/topic_and_service_names.html) describes name resolution.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
